# 01_run_bronze_ingest

Runs the W2 ingestion framework in the coud and verifies Bronze against
the landing subset. The only thing that changes from local is the injected
environment; framework code (bronze.py, run.py) is untouched

In [0]:
# The notebook cwd is NOT the repo root, so run.py's glob (Path("config/sources"))
# would come back empty. chdir anchors it; sys.path.insert lets the src package import.
import os, sys

REPO = "/Workspace/Users/joseluis.estr@gmail.com/gaming-telemetry-platform"
os.chdir(REPO)
sys.path.insert(0, REPO)

In [0]:
# Run the exact same ingestion as local, injecting the cloud environment.
# main() takes an Environment; passing DatabricksEnvironment() is the only thing
# that changes between local and cloud. bronze.py and run.py stay untouched.
from src.ingestion.run import main
from src.ingestion.environment import DatabricksEnvironment

main(DatabricksEnvironment())

In [0]:
# Bronze row count must equal the landing row count for this subset: Bronze keeps
# duplicates by design (row-level dedup is Silver's job), so nothing is dropped here.
# The per-day breakdown confirms the partitions landed even, which is the W1 design:
# skew lives on game_id, not on event_date.
spark.sql("SELECT COUNT(*) FROM workspace.telemetry.bronze_player_events").show()
spark.sql(
    "SELECT event_date, COUNT(*) "
    "FROM workspace.telemetry.bronze_player_events "
    "GROUP BY event_date ORDER BY event_date"
).show()

In [0]:
from pyspark.sql.functions import col
 
# The Delta transaction log is the authoritative per-run record. numInputRows lied
# (None on serverless AvailableNow); operationMetrics.numOutputRows is what each
# commit actually wrote. Same append-only log that makes the ingest idempotent. DDIA Ch 3.
(spark.sql("DESCRIBE HISTORY workspace.telemetry.bronze_player_events")
   .select("version", "operation",
           col("operationMetrics")["numOutputRows"].alias("rows_written"))
   .orderBy("version")
   .show(truncate=False))

In [0]:
# Confirm delta.autoOptimize.optimizeWrite is NOT set. If it were, the "before" file
# count below would already be compacted and the OPTIMIZE delta would be a lie.
spark.sql("SHOW TBLPROPERTIES workspace.telemetry.bronze_player_events").show(truncate=False)

In [0]:
# Count files in the landing (the source of the small-files tax), not in Bronze.
# Bronze arrives already packed because serverless compacts the Autoloader write.
landing = "/Volumes/workspace/telemetry/landing"
files = dbutils.fs.ls(f"{landing}/event_date=2026-01-14")
print(f"landing 01-14: {len(files)} files")

In [0]:
from delta.tables import DeltaTable
 
dt = DeltaTable.forName(spark, "workspace.telemetry.bronze_player_events")
 
# Capture before, then OPTIMIZE Bronze. Bronze already arrived compacted (~6 files)
# because serverless packs on write, so before/after will barely move. That is the
# finding, not a failure: the small-files tax lives in the landing, not in Bronze.
before = dt.detail().select("numFiles", "sizeInBytes").first()
 
result = dt.optimize().executeCompaction()
 
after = dt.detail().select("numFiles", "sizeInBytes").first()
print(f"before: {before['numFiles']} files, {before['sizeInBytes']} bytes")
print(f"after:  {after['numFiles']} files, {after['sizeInBytes']} bytes")

In [0]:
# Inspect the OPTIMIZE commit: numRemovedFiles / numAddedFiles show what bin-packing
# actually merged, bounded by the event_date partition. DDIA Ch 6: the partition that
# gives read pruning is the same boundary that bounds compaction.
(spark.sql("DESCRIBE HISTORY workspace.telemetry.bronze_player_events")
   .select("version", "operation", "operationMetrics")
   .filter("operation = 'OPTIMIZE'")
   .show(truncate=False))